In [ ]:
import msprime, tskit
import numpy as np
import gaiapy as gp
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from tqdm import tqdm


In [ ]:
def old_get_span_stats(ts, ets):
    time_map = {}
    added_span = np.zeros(ets.num_nodes)
    wrong_added_span = np.zeros(ets.num_nodes)
    for n in ts.nodes():
        # check times are unique except for samples
        assert n.time == 0.0 or n.time not in time_map
        time_map[n.time] = n.id
    # total_added_span = 0
    # wrongly_added_span = 0
    print("made past time map")
    for interval, t, et in ts.coiterate(ets):
        interval_length = interval[1] - interval[0]
        t_nodes = list(t.nodes())
        for n in et.nodes():
            if et.num_children(n) == 1:
                added_span[n] += interval_length
                # total_added_span += interval_length
            on = time_map[et.time(n)]
            if on not in t_nodes:
                assert et.num_children(n) == 1
                wrong_added_span[n] += interval_length
                # wrongly_added_span += interval_length
    return added_span, wrong_added_span

def node_spans(ts, include_missing=False):
    """
    Returns the array of "node spans", i.e., the `j`th entry gives
    the total span over which node `j` is in the tree sequence.
    Sample nodes that are isolated are "missing data"; inclusion
    of these spans are controlled by `include_missing`. (If
    `include_missing` is `True` then the span of each sample is
    always equal to the sequence length.)

    :param bool include_missing: Whether to include spans of nodes
        on which they have missing data.
    """
    child_spans = np.bincount(
        ts.edges_child,
        weights=ts.edges_right - ts.edges_left,
        minlength=ts.num_nodes,
    )
    for t in ts.trees():
        span = t.span
        for r in t.roots:
            # do this check to exempt 'missing data'
            if include_missing or (t.num_children(r) > 0):
                child_spans[r] += span
    return child_spans

def findUnary(ts):
    unary_nodes = np.zeros(ts.num_nodes) # binary vector specifying if a node is unary or not anywhere on the tree sequence
    for tree in ts.trees():
        num_children = tree.num_children_array[:ts.num_nodes]
        is_unary = num_children == 1
        for i, condition in enumerate(is_unary):
            if is_unary[i] == True:
                unary_nodes[i] = 1
    mask = unary_nodes == 1
    #mask, unary_nodes[mask]
    unary_list = np.where(mask)
    unary_indices = unary_list[0]
    return unary_indices

def small_getAccOut(ts, samples, ancestors, everything):
    simp = ts.simplify()
    ext = ts.extend_haplotypes()
    unary_indices = findUnary(ext)
    print("1")

    total_added_span, wrongly_added_span = get_span_stats(ts, ext)

    print("1.5")

    simp_spans = node_spans(simp)
    ext_spans = node_spans(ext)

    print("2")

    simp_num_trees = simp.num_trees
    ext_num_trees = ext.num_trees
    simp_num_edges = simp.num_edges
    ext_num_edges = ext.num_edges

    print("3")
    # locs = locations(ext)
    sample_locations = samples # comment this out when not usign test data set 
    ancestor_locations = ancestors

    sample_centroid = np.mean(sample_locations[:, 1:], axis=0)
    # print(sample_centroid)

    print("4")

    simp_mpr = gp.quadratic_mpr(simp, sample_locations)
    simp_map_x = gp.quadratic_mpr_minimize(simp_mpr)

    print("5")

    ext_mpr = gp.quadratic_mpr(ext, sample_locations)
    ext_map_x = gp.quadratic_mpr_minimize(ext_mpr)

    print("6")

    is_unary = np.isin(everything[:, 0], unary_indices)
    is_ancestor = np.isin(everything[:, 0], unary_indices)
    target_nodes = is_unary & is_ancestor 
    target_locations = everything[target_nodes][:, [0, 1, 2]]

    print("7")

    simp_e = np.sqrt(np.sum((simp_map_x[target_nodes] - target_locations[:, 1:2])**2, axis=1)) / np.max(pdist(sample_locations[:, 1:2]))
    ext_e = np.sqrt(np.sum((ext_map_x[target_nodes] - target_locations[:, 1:2])**2, axis=1)) / np.max(pdist(sample_locations[:, 1:2]))

    print("8")

    dist_from_sample_centroid0 = np.sqrt(np.sum((target_locations[:, 1:3] - sample_centroid)**2, axis=1))

    simp_dist_from_sample_centroid = np.sqrt(np.sum((simp_map_x[target_nodes] - sample_centroid)**2, axis=1))
    ext_dist_from_sample_centroid = np.sqrt(np.sum((ext_map_x[target_nodes] - sample_centroid)**2, axis=1))

    print("9")

    target_node_ids = everything[target_nodes, 0]
    target_node_times = ext.nodes_time[target_nodes]

    print("10")

    target_simp_spans = simp_spans[target_nodes]
    target_ext_spans = ext_spans[target_nodes]

    print("11")

    target_total_added_span = total_added_span[target_nodes]
    target_wrong_added_span = wrongly_added_span[target_nodes]

    print("12")


    ancestor_df = pd.DataFrame({
        'node_id': target_node_ids,
        'node_time': target_node_times,
        'simp_error': simp_e,
        'ext_error': ext_e,
        'simp_span': target_simp_spans,
        'ext_span': target_ext_spans,
        'added_span': target_total_added_span,
        'wrongly_added_span': target_wrong_added_span,
        'dist_from_sample_centroid0': dist_from_sample_centroid0,
        'simp_dist_from_sample_centroid': simp_dist_from_sample_centroid,
        'ext_dist_from_sample_centroid': ext_dist_from_sample_centroid
    })

    print("13")

    static_df = pd.DataFrame({
        'simp_num_trees': [simp_num_trees],
        'ext_num_trees': [ext_num_trees], 
        'simp_num_edges': [simp_num_edges],
        'ext_num_edges': [ext_num_edges]
    })
    print("14")

    # ancestor_df.to_csv(f"shits.csv",
    #                   mode='a', header=True, index=False)
    
    # static_df.to_csv(f"shits_static_info.csv",
    #                   mode='a', header=True, index=False)
    
    return ancestor_df, static_df

def locations(ts):
    nodes = ts.nodes()
    locs_array = []
    for node in nodes:
        if node.individual != -1:
            ind = ts.individual(node.individual)
            x = ind.location[0]
            y = ind.location[1]
            is_sample = node.is_sample()
            locs_array.append(node.id)
            locs_array.append(is_sample)
            locs_array.append(x)
            locs_array.append(y)
        
    locs_array = np.array(locs_array)
    locs = locs_array.reshape(-1, 4)
    return locs

def getAccOut(ts, outPrefix, sigma, rep):


    # tqdm - package to test timing of things 
    simp = ts.simplify()
    ext = ts.extend_haplotypes()
    unary_indices = findUnary(simp)

    total_added_span, wrongly_added_span = get_span_stats(ts, ext)
    
    simp_spans = node_spans(simp)
    ext_spans = node_spans(ext)

    simp_num_trees = simp.num_trees
    ext_num_trees = ext.num_trees
    simp_num_edges = simp.num_edges
    ext_num_edges = ext.num_edges

    locs = locations(ext)
    sample_locations = locs[locs[:, 1] == 1][:, [0, 2, 3]]
    ancestor_locations = locs[locs[:, 1] != 1][:, [0, 2, 3]]


    sample_centroid = np.mean(sample_locations[:, 1:], axis=0)
    # print(sample_centroid)

    simp_mpr = gp.quadratic_mpr(simp, sample_locations)
    simp_map_x = gp.quadratic_mpr_minimize(simp_mpr)

    ext_mpr = gp.quadratic_mpr(ext, sample_locations)
    ext_map_x = gp.quadratic_mpr_minimize(ext_mpr)
    
    is_unary = np.isin(locs[:, 0], unary_indices)
    is_ancestor = np.isin(locs[:, 0], unary_indices)
    target_nodes = is_unary & is_ancestor 
    target_locations = locs[target_nodes][:, [0, 2, 3]]

    simp_e = np.sqrt(np.sum((simp_map_x[target_nodes] - target_locations[:, 1:2])**2, axis=1)) / np.max(pdist(sample_locations[:, 1:2]))
    ext_e = np.sqrt(np.sum((ext_map_x[target_nodes] - target_locations[:, 1:2])**2, axis=1)) / np.max(pdist(sample_locations[:, 1:2]))
    
    dist_from_sample_centroid0 = np.sqrt(np.sum((target_locations[:, 1:3] - sample_centroid)**2, axis=1))

    simp_dist_from_sample_centroid = np.sqrt(np.sum((simp_map_x[target_nodes] - sample_centroid)**2, axis=1))
    ext_dist_from_sample_centroid = np.sqrt(np.sum((ext_map_x[target_nodes] - sample_centroid)**2, axis=1))
    
    target_node_ids = locs[target_nodes, 0]
    target_node_times = ext.nodes_time[target_nodes]

    target_simp_spans = simp_spans[target_nodes]
    target_ext_spans = ext_spans[target_nodes]

    target_total_added_span = total_added_span[target_nodes]
    target_wrong_added_span = wrongly_added_span[target_nodes]
   

    ancestor_df = pd.DataFrame({
        'node_id': target_node_ids,
        'node_time': target_node_times,
        'simp_error': simp_e,
        'ext_error': ext_e,
        'simp_span': target_simp_spans,
        'ext_span': target_ext_spans,
        'added_span': target_total_added_span,
        'wrongly_added_span': target_wrong_added_span,
        'dist_from_sample_centroid0': dist_from_sample_centroid0,
        'simp_dist_from_sample_centroid': simp_dist_from_sample_centroid,
        'ext_dist_from_sample_centroid': ext_dist_from_sample_centroid
    })

    static_df = pd.DataFrame({
        'sigma': sigma,
        'rep': rep,
        'simp_num_trees': simp_num_trees,
        'ext_num_trees': ext_num_trees, 
        'simp_num_edges': simp_num_edges,
        'ext_num_edges': ext_num_edges
    })

    ancestor_df.to_csv(f"{outPrefix}_results.csv",
                      mode='a', header=True, index=False)
    
    static_df.to_csv(f"{outPrefix}_static_info.csv",
                      mode='a', header=True, index=False)
    
    return ancestor_df, static_df

def get_node_stats(ts):
    #nodes = list(ts.nodes())
    nodes = np.array(list(ts.nodes()))
    node_ids = np.array([n.id for n in nodes])
    # Find the samples
    #is_sample = np.asarray(np.isin(nodes, ts.samples()), dtype=int)
    is_sample = np.asarray(np.isin(node_ids, ts.samples()), dtype=int)
    # Find all the other things (this requires checking tree by tree)
    tree = ts.first()
    start = tree.interval[0] 
    end = ts.sequence_length
    num_children = np.zeros(nodes.shape[0])
    num_parents = np.zeros(nodes.shape[0])
    distinct_children, distinct_parents, distinct_populations = list(), list(), list()
    is_root = np.zeros(nodes.shape[0])
    #for i,node in tqdm(enumerate(nodes)):
    for i, node_id in tqdm(enumerate(node_ids)): 
        tree.seek(start)
        children = list()
        parents = list()
        is_root[i] = tree.is_root(node_id)
        node_children = list(tree.children(node_id))

        children.extend(node_children)
        parents.append(tree.parent(node_id))

        w = (None, tree.interval[0])
        while w[1] < end and tree.next():
            w = (w[1], min(tree.interval[1], end))
            is_root[i] = tree.is_root(node_id)
            node_children = list(tree.children(node_id))

            children.extend(node_children)
            parents.append(tree.parent(node_id))

        distinct_parents.append(np.unique(parents))
        distinct_children.append(np.unique(children))
        num_children[i] = np.unique(children).shape[0]
        num_parents[i] = np.unique(parents).shape[0]
    data_dict = {
        'id': node_ids,
        'num_children': num_children,
        'distinct_children': distinct_children,
        'distinct_parents': distinct_parents,
        'num_parents': num_parents,
        'is_sample': is_sample,
        'is_root': is_root,
    }
    return pd.DataFrame(data_dict)

def get_span_stats(ts, ets):
    added_span = np.zeros(ets.num_nodes)
    wrong_added_span = np.zeros(ets.num_nodes)
    
    ts_node_ids = set(n.id for n in ts.nodes())  # all valid node IDs in original ts

    for interval, t, et in ts.coiterate(ets):
        interval_length = interval[1] - interval[0]
        t_nodes = set(t.nodes())  # node IDs present in this specific tree
        for n in et.nodes():
            if et.num_children(n) == 1:
                added_span[n] += interval_length
            if n not in t_nodes:
                assert et.num_children(n) == 1
                wrong_added_span[n] += interval_length

    return added_span, wrong_added_span

In [ ]:
def get_span_stats(ts, ets):
    added_span = np.zeros(ets.num_nodes)
    wrong_added_span = np.zeros(ets.num_nodes)
    
    ts_node_ids = set(n.id for n in ts.nodes())  # all valid node IDs in original ts

    for interval, t, et in ts.coiterate(ets):
        interval_length = interval[1] - interval[0]
        t_nodes = set(t.nodes())  # node IDs present in this specific tree
        for n in et.nodes():
            if et.num_children(n) == 1:
                added_span[n] += interval_length
            if n not in t_nodes:
                assert et.num_children(n) == 1
                wrong_added_span[n] += interval_length

    return added_span, wrong_added_span

In [ ]:
test = msprime.sim_ancestry(
    samples=6,            # 6 diploid individuals = 12 haploid chromosomes
    population_size=500,
    sequence_length=10_000,
    recombination_rate=1e-7,   # low recomb → likely a single tree
    random_seed=42,
)
stest = test.simplify()
etest = stest.extend_haplotypes()
get_span_stats(test, etest)

In [ ]:
old_get_span_stats(test, etest)

In [ ]:
ts = tskit.load("/home/islar/bradburdlab/tree_project/unary_project/tree-S0.5-R3.trees")
simp = ts.simplify()
ext = simp.extend_haplotypes()

In [ ]:
ts8 = tskit.load("/home/islar/bradburdlab/tree_project/unary_project/tree-S0.8-R8.trees")
simp8 = ts.simplify()
ets8 = simp.extend_haplotypes()

In [ ]:
get_span_stats(ts, ext)

In [ ]:
get_span_stats(simp, ext)
# total_added_span, wrongly_added_span = get_span_stats(simp, ext)
# total_added_span
# print(f"Out of a total of {total_added_span} added edge span, "
#       f"we have wrongly added {wrongly_added_span} span, "
#       f"a proportion of {wrongly_added_span / total_added_span}.")

In [ ]:
def example4():        
    node_times = (0, 0, 0, 0, 1, 1, 3, 2, 2)
    samples = (0, 1, 2, 3)
    # (p, c, l, r)
    extended_edges = [
        (4, 0, 0, 10),
        (4, 1, 0, 5),
        (4, 1, 7, 10),
        (5, 2, 0, 2),
        (5, 2, 5, 10),
        (5, 3, 0, 10),
        (7, 2, 2, 5),
        (7, 4, 0, 10),
        (8, 1, 5, 7),
        (8, 5, 0, 10),
        (6, 7, 0, 10),
        (6, 8, 0, 10),
    ]
    edges = [
        (4, 0, 0, 10),
        (4, 1, 0, 5),
        (4, 1, 7, 10),
        (5, 2, 0, 2),
        (5, 2, 5, 10),
        (5, 3, 0, 2),
        (5, 3, 5, 10),
        (7, 2, 2, 5),
        (7, 4, 2, 5),
        (8, 1, 5, 7),
        (8, 5, 5, 7),
        (6, 3, 2, 5),
        (6, 4, 0, 2),
        (6, 4, 5, 10),
        (6, 5, 0, 2),
        (6, 5, 7, 10),
        (6, 7, 2, 5),
        (6, 8, 5, 7),
    ]
    tables = tskit.TableCollection(sequence_length=10)
    tables.sort()
    for n, t in enumerate(node_times):
        flags = tskit.NODE_IS_SAMPLE if n in samples else 0
        tables.nodes.add_row(time=t, flags=flags)
    for p, c, l, r in edges:
        tables.edges.add_row(parent=p, child=c, left=l, right=r)
    ts = tables.tree_sequence()
    tables.edges.clear()
    for p, c, l, r in extended_edges:
        tables.edges.add_row(parent=p, child=c, left=l, right=r)
    ets = tables.tree_sequence()
    assert ts.num_edges == 18
    assert ets.num_edges == 12
    return ts, ets

t4, et4 = example4()

samples4 = np.array([
    [0, 1.5, 2.0],  # node 0 at coordinates (1.5, 2.0)
    [1, 4.2, 3.1],  # node 1 at coordinates (4.2, 3.1) 
    [2, 6.6, 5.5],  # node 2 at coordinates (6.7, 5.5)
    [3, 6.8, 5.5],  # node 2 at coordinates (6.7, 5.5)
])

ancestors4 = np.array([
    [4, 6.9, 5.5],  # node 2 at coordinates (6.7, 5.5)
    [5, 7, 5.5],  # node 2 at coordinates (6.7, 5.5)
    [6, 2, 4.0],  # node 0 at coordinates (1.5, 2.0)
    [7, 4.1, 3.9],  # node 1 at coordinates (4.2, 3.1) 
    [8, 6.0, 5.4],  # node 2 at coordinates (6.7, 5.5)
 ])

everything4 = np.array([
    [0, 1.5, 2.0],  # node 0 at coordinates (1.5, 2.0)
    [1, 4.2, 3.1],  # node 1 at coordinates (4.2, 3.1) 
    [2, 6.6, 5.5],  # node 2 at coordinates (6.7, 5.5)
    [3, 6.8, 5.5],  # node 2 at coordinates (6.7, 5.5)
    [4, 6.9, 5.5],  # node 2 at coordinates (6.7, 5.5)
    [5, 7, 5.5],  # node 2 at coordinates (6.7, 5.5)
    [6, 2, 4.0],  # node 0 at coordinates (1.5, 2.0)
    [7, 4.1, 3.9],  # node 1 at coordinates (4.2, 3.1) 
    [8, 6.0, 5.4],  # node 2 at coordinates (6.7, 5.5)
])

s4 = t4.simplify()

# get_span_stats(t4, et4)
# old_get_span_stats(t4, et4)
# total_added_span, wrongly_added_span = get_span_stats(t4, et4)
# total_added_span
# print(f"Out of a total of {total_added_span} added edge span, "
#       f"we have wrongly added {wrongly_added_span} span, "
#       f"a proportion of {wrongly_added_span / total_added_span}.")

In [ ]:
get_node_stats(t4)

In [ ]:
t4.draw_svg()

In [ ]:
get_node_stats(t4)

In [ ]:
def example2():
        node_times = {
            0: 0,
            1: 0,
            2: 0,
            3: 0,
            4: 0,
            5: 0,
            6: 0,
            7: 0,
            8: 0,
            9: 0,
            10: 1,
            11: 2,
            12: 3,
            13: 4,
            14: 5,
            15: 6,
            16: 7,
            17: 8,
            18: 9,
            19: 10,
            20: 11,
            21: 12,
        }
        # (p,c,l,r)
        edges = [
            (10, 2, 0, 9),
            (10, 5, 0, 9),
            (11, 0, 0, 9),
            (11, 7, 0, 9),
            (12, 3, 3, 9),
            (12, 9, 3, 9),
            (13, 4, 0, 9),
            (13, 11, 0, 9),
            (14, 6, 0, 9),
            (14, 10, 0, 9),
            (15, 9, 0, 3),
            (15, 13, 0, 3),
            (16, 1, 0, 6),
            (16, 3, 0, 3),
            (16, 12, 3, 6),
            (17, 1, 6, 9),
            (17, 13, 6, 9),
            (18, 13, 3, 6),
            (18, 14, 0, 9),
            (18, 15, 0, 3),
            (18, 17, 6, 9),
            (19, 8, 0, 9),
            (19, 12, 6, 9),
            (19, 16, 0, 6),
            (20, 18, 0, 3),
            (20, 18, 6, 9),
            (20, 19, 0, 3),
            (20, 19, 6, 9),
            (21, 18, 3, 6),
            (21, 19, 3, 6),
        ]
        extended_edges = [
            (10, 2, 0.0, 9.0),
            (10, 5, 0.0, 9.0),
            (11, 0, 0.0, 9.0),
            (11, 7, 0.0, 9.0),
            (12, 3, 0.0, 9.0),
            (12, 9, 3.0, 9.0),
            (13, 4, 0.0, 9.0),
            (13, 11, 0.0, 9.0),
            (14, 6, 0.0, 9.0),
            (14, 10, 0.0, 9.0),
            (15, 9, 0.0, 3.0),
            (15, 13, 0.0, 9.0),
            (16, 1, 0.0, 6.0),
            (16, 12, 0.0, 9.0),
            (17, 1, 6.0, 9.0),
            (17, 15, 0.0, 9.0),
            (18, 14, 0.0, 9.0),
            (18, 17, 0.0, 9.0),
            (19, 8, 0.0, 9.0),
            (19, 16, 0.0, 9.0),
            (20, 18, 0.0, 3.0),
            (20, 18, 6.0, 9.0),
            (20, 19, 0.0, 3.0),
            (20, 19, 6.0, 9.0),
            (21, 18, 3.0, 6.0),
            (21, 19, 3.0, 6.0),
        ]
        samples = list(np.arange(10))
        tables = tskit.TableCollection(sequence_length=9)
        for (
            n,
            t,
        ) in node_times.items():
            flags = tskit.NODE_IS_SAMPLE if n in samples else 0
            tables.nodes.add_row(time=t, flags=flags)
        for p, c, l, r in edges:
            tables.edges.add_row(parent=p, child=c, left=l, right=r)
        ts = tables.tree_sequence()
        tables.edges.clear()
        for p, c, l, r in extended_edges:
            tables.edges.add_row(parent=p, child=c, left=l, right=r)
        ets = tables.tree_sequence()
        assert ts.num_edges == 30
        assert ets.num_edges == 26
        return ts, ets

samples2 = np.array([
    [0, 1.5, 2.0],  # node 0 at coordinates (1.5, 2.0)
    [1, 4.2, 3.1],  # node 1 at coordinates (4.2, 3.1) 
    [2, 6.6, 5.5],  # node 2 at coordinates (6.7, 5.5)
    [3, 6.8, 5.5],  # node 2 at coordinates (6.7, 5.5) 
    [4, 9.0, 10.5],  # node 2 at coordinates (6.7, 5.5) 
    [5, 11.3, 2.5],  # node 2 at coordinates (6.7, 5.5) 
    [6, 11.2, 5.4],  # node 2 at coordinates (6.7, 5.5) 
    [7, 12.0, 6.9],  # node 2 at coordinates (6.7, 5.5) 
    [8, 11.1, 4.7],  # node 2 at coordinates (6.7, 5.5) 
    [9, 1.6, 2.0],  # node 0 at coordinates (1.5, 2.0) 
])

ancestors2 = np.array([
    [10, 2, 2.1],  # node 0 at coordinates (1.5, 2.0)
    [11, 3.6, 5.2],  # node 0 at coordinates (1.5, 2.0)
    [12, 4.1, 7.8],  # node 0 at coordinates (1.5, 2.0)
    [13, 7.8, 8.9],  # node 0 at coordinates (1.5, 2.0)
    [14, 5.6, 2.0],  # node 0 at coordinates (1.5, 2.0)
    [15, 9.7, 3.1],  # node 0 at coordinates (1.5, 2.0)
    [16, 9.8, 11.1],  # node 0 at coordinates (1.5, 2.0)
    [17, 10.1, 11.6],  # node 0 at coordinates (1.5, 2.0)
    [18, 11.2, 2.1],  # node 0 at coordinates (1.5, 2.0)
    [19, 14.1, 4.0],  # node 0 at coordinates (1.5, 2.0)
    [20, 1.2, 8.9],  # node 0 at coordinates (1.5, 2.0)
    [21, 4.3, 8.7],  # node 0 at coordinates (1.5, 2.0)
])

everything2 = np.array([
    [0, 1.5, 2.0],  # node 0 at coordinates (1.5, 2.0)
    [1, 4.2, 3.1],  # node 1 at coordinates (4.2, 3.1) 
    [2, 6.6, 5.5],  # node 2 at coordinates (6.7, 5.5)
    [3, 6.8, 5.5],  # node 2 at coordinates (6.7, 5.5) 
    [4, 9.0, 10.5],  # node 2 at coordinates (6.7, 5.5) 
    [5, 11.3, 2.5],  # node 2 at coordinates (6.7, 5.5) 
    [6, 11.2, 5.4],  # node 2 at coordinates (6.7, 5.5) 
    [7, 12.0, 6.9],  # node 2 at coordinates (6.7, 5.5) 
    [8, 11.1, 4.7],  # node 2 at coordinates (6.7, 5.5) 
    [9, 1.6, 2.0],  # node 0 at coordinates (1.5, 2.0) 
    [10, 2, 2.1],  # node 0 at coordinates (1.5, 2.0)
    [11, 3.6, 5.2],  # node 0 at coordinates (1.5, 2.0)
    [12, 4.1, 7.8],  # node 0 at coordinates (1.5, 2.0)
    [13, 7.8, 8.9],  # node 0 at coordinates (1.5, 2.0)
    [14, 5.6, 2.0],  # node 0 at coordinates (1.5, 2.0)
    [15, 9.7, 3.1],  # node 0 at coordinates (1.5, 2.0)
    [16, 9.8, 11.1],  # node 0 at coordinates (1.5, 2.0)
    [17, 10.1, 11.6],  # node 0 at coordinates (1.5, 2.0)
    [18, 11.2, 2.1],  # node 0 at coordinates (1.5, 2.0)
    [19, 14.1, 4.0],  # node 0 at coordinates (1.5, 2.0)
    [20, 1.2, 8.9],  # node 0 at coordinates (1.5, 2.0)
    [21, 4.3, 8.7],  # node 0 at coordinates (1.5, 2.0)
])

t2, et2 = example2()

get_span_stats(t2, et2)


#small_getAccOut(t2, samples2, ancestors2, everything2)
# total_added_span, wrongly_added_span = get_span_stats(t2, et2)
# total_added_span
# print(f"Out of a total of {total_added_span} added edge span, "
#       f"we have wrongly added {wrongly_added_span} span, "
#       f"a proportion of {wrongly_added_span / total_added_span}.")


In [ ]:
old_get_span_stats(t2, et2)

In [ ]:
def example1():
        node_times = {
            0: 0,
            1: 0,
            2: 0,
            3: 0,
            4: 1,
            5: 1,
            6: 4,
            7: 6,
            8: 10,
            9: 4,
            10: 12,
            11: 8,
            12: 8,
            13: 15,
        }
        # (p,c,l,r)
        edges = [
            (4, 0, 0, 9),
            (4, 1, 0, 9),
            (5, 2, 0, 6),
            (5, 3, 0, 9),
            (6, 4, 0, 3),
            (9, 5, 0, 3),
            (7, 4, 3, 6),
            (11, 7, 3, 6),
            (12, 5, 3, 6),
            (8, 2, 6, 9),
            (8, 4, 6, 9),
            (8, 6, 0, 3),
            (10, 5, 6, 9),
            (10, 8, 0, 3),
            (10, 8, 6, 9),
            (10, 9, 0, 3),
            (10, 11, 3, 6),
            (10, 12, 3, 6),
            (13, 10, 3, 6),
        ]
        extended_edges = [
            (4, 0, 0.0, 9.0),
            (4, 1, 0.0, 9.0),
            (5, 2, 0.0, 6.0),
            (5, 3, 0.0, 9.0),
            (6, 4, 0.0, 9.0),
            (9, 5, 0.0, 9.0),
            (7, 6, 0.0, 9.0),
            (11, 7, 0.0, 9.0),
            (12, 9, 0.0, 9.0),
            (8, 2, 6.0, 9.0),
            (8, 11, 0.0, 9.0),
            (10, 8, 0.0, 9.0),
            (10, 12, 0.0, 9.0),
            (13, 10, 3.0, 6.0),
        ]
        samples = list(np.arange(4))
        tables = tskit.TableCollection(sequence_length=9)
        for (
            n,
            t,
        ) in node_times.items():
            flags = tskit.NODE_IS_SAMPLE if n in samples else 0
            tables.nodes.add_row(time=t, flags=flags)
        for p, c, l, r in edges:
            tables.edges.add_row(parent=p, child=c, left=l, right=r)
        ts = tables.tree_sequence()
        tables.edges.clear()
        for p, c, l, r in extended_edges:
            tables.edges.add_row(parent=p, child=c, left=l, right=r)
        ets = tables.tree_sequence()
        assert ts.num_edges == 19
        assert ets.num_edges == 14
        return ts, ets

t1, et1 = example1()


In [ ]:
def attempt1():
    node_times = (0, 0, 0, 1, 2, 3)
    samples = (0, 1, 2)
    # (p, c, l, r)
    extended_edges = [
        (3, 0, 0, 4),
        (3, 1, 0, 3), 
        (4, 1, 3, 4),
        (4, 2, 0, 4), 
        (5, 3, 0, 4), 
        (5, 4, 0, 4),
    ]
    edges = [
        (3, 0, 0, 3),
        (3, 1, 0, 3), 
        (4, 1, 3, 4),
        (4, 2, 3, 4), 
        (5, 0, 3, 4),
        (5, 2, 0, 3), 
        (5, 3, 0, 3), 
        (5, 4, 3, 4),
    ]
    tables = tskit.TableCollection(sequence_length=4)
    tables.sort()
    for n, t in enumerate(node_times):
        flags = tskit.NODE_IS_SAMPLE if n in samples else 0
        tables.nodes.add_row(time=t, flags=flags)
    for p, c, l, r in edges:
        tables.edges.add_row(parent=p, child=c, left=l, right=r)
    ts = tables.tree_sequence()
    tables.edges.clear()
    for p, c, l, r in extended_edges:
        tables.edges.add_row(parent=p, child=c, left=l, right=r)
    ets = tables.tree_sequence()
    assert ts.num_edges == 8
    assert ets.num_edges == 6
    return ts, ets

t5, et5 = attempt1()
